# Camada Silver

Este notebook executa os tratamentos das cinco fontes da Bronze e grava Parquets em `data/silver`. Execute as células em ordem com o kernel do ambiente virtual do projeto.

Cada função valida os dados antes de gravar e substitui a saída anterior daquela fonte. As pastas usam siglas como `SP`, com nomes legíveis para os arquivos. Não são gerados marcadores `_SUCCESS`; os arquivos ocultos `.crc` acompanham os Parquets.

As saídas preservam a origem dos registros. Campos vazios e marcadores de ausência são tratados conforme cada fonte. Alertas físicos de clima ficam em `quality_flags`; valores extremos não são removidos automaticamente. Não há preenchimento de lacunas ou integração entre INMET e Open-Meteo nesta etapa.

As novas saídas são gravadas em uma pasta auxiliar e validadas antes de substituir a carga anterior. Não se mantém histórico de pastas após uma execução bem-sucedida.

A gravação usa uma trava do sistema operacional e um diário de recuperação. Após encerramento brusco do processo, a próxima gravação recupera a operação pendente automaticamente. Para recuperar antes de consultar, use `pipeline.shared.storage.recover_output(caminho)`. O pequeno arquivo `.write-lock` é reutilizado e não precisa ser apagado.


O acompanhamento usa somente o estado atual. Os valores permanecem nos arquivos das camadas; não são criados snapshots. Para atualizar os vínculos no PostgreSQL, execute a Gold e depois o Load.


## 1. Imports e localização do projeto e definir os caminhos

O ano seleciona as pastas de clima. A data de referência da população é extraída do título preservado na Bronze, independentemente desta configuração.

In [ ]:
import os
import sys
from pathlib import Path
from importlib import reload

from pyspark.sql import SparkSession
from pyspark.sql import functions as F


project_directory = Path.cwd().resolve()

# Procura a raiz a partir da pasta atual, sem depender de __file__.
while not (project_directory / "pipeline" / "silver").is_dir() or not (project_directory / "data").is_dir():

    if project_directory == project_directory.parent:
        raise FileNotFoundError("Abra o notebook com o diretório de trabalho dentro do projeto Viajando_com_o_dado.")

    project_directory = project_directory.parent

# Permite importar explicitamente pipeline.silver, mesmo com a Bronze no sys.path.
sys.path.insert(0, str(project_directory))

from pipeline.shared import execution
from pipeline.silver import geocoding
from pipeline.silver import ibge_municipalities
from pipeline.silver import ibge_population
from pipeline.silver import inmet
from pipeline.silver import open_meteo


DATA_DIRECTORY = project_directory / "data"
BRONZE_DIRECTORY = DATA_DIRECTORY / "bronze"
SILVER_DIRECTORY = DATA_DIRECTORY / "silver"
YEAR = "2025"

local_filesystem_jar = project_directory / "pipeline" / "bronze" / "lib" / "hadoop-bare-naked-local-fs-0.1.0.jar"
required_paths = [
    BRONZE_DIRECTORY / "ibge" / "municipalities",
    BRONZE_DIRECTORY / "ibge" / "population",
    BRONZE_DIRECTORY / "open_meteo" / "geocoding",
    BRONZE_DIRECTORY / "open_meteo" / "weather" / YEAR,
    BRONZE_DIRECTORY / "inmet" / YEAR,
    local_filesystem_jar,
]

for required_path in required_paths:

    if not required_path.exists():
        raise FileNotFoundError(f"Caminho necessário não encontrado: {required_path}")

print(f"Projeto: {project_directory}")
print(f"Entrada: {BRONZE_DIRECTORY}")
print(f"Destino: {SILVER_DIRECTORY}")
print(f"Ano das séries meteorológicas: {YEAR}")


In [ ]:
session = execution.run_context(project_directory, parameters={"year": str(globals().get("YEAR", "2025"))})
state = session.__enter__()
print("Acompanhando a execução atual.")


## 2. Importar os módulos

Os imports usam `pipeline.silver` para distinguir estas funções das ingestões da Bronze e da coleta de APIs. Use um kernel novo ao começar a execução.

O INMET usa `timezonefinder` para identificar o fuso pelas coordenadas. As dependências do projeto estão em `requirements.txt` e já foram instaladas no ambiente virtual deste projeto.

## 3. Iniciar o Spark

A sessão é compartilhada pelas cinco transformações, com duas tarefas simultâneas. Mantemos a biblioteca de acesso local já usada no Windows. Reinicie o kernel após alterar a configuração do Spark.

O INMET usa partições menores e etapas intermediárias em disco para processar a base horária. Sua execução pode levar alguns minutos e usa espaço temporário em disco. A sessão exibe instantes em UTC; as datas diárias da Open-Meteo continuam no calendário local informado na resposta.

In [ ]:
os.environ["PYSPARK_PYTHON"] = sys.executable

spark = (
    SparkSession.builder
    .appName("Viajando com o dado - Silver")
    .master("local[2]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.hadoop.mapreduce.fileoutputcommitter.marksuccessfuljobs", "false")
    .config("spark.pyspark.python", sys.executable)
    .config("spark.driver.extraClassPath", str(local_filesystem_jar))
    .config("spark.executor.extraClassPath", str(local_filesystem_jar))
    .config("spark.hadoop.fs.file.impl", "com.globalmentor.apache.hadoop.fs.BareLocalFileSystem")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
print(f"Versão do Spark: {spark.version}")


## 4. Municípios do IBGE

Abre as hierarquias em colunas, padroniza textos e valida ID e UF. Mantém as divisões de microrregião/mesorregião e de região imediata/intermediária separadas. A coleta atual contém as 27 capitais.

Entrada: `data/bronze/ibge/municipalities`  
Saída: `data/silver/ibge/municipalities`

In [ ]:
# Recarrega o código salvo para não usar uma versão anterior mantida no kernel.
reload(ibge_municipalities)

municipalities = ibge_municipalities.transform_municipalities(
    spark,
    BRONZE_DIRECTORY / "ibge" / "municipalities",
    SILVER_DIRECTORY / "ibge" / "municipalities",
)

municipalities.printSchema()
municipalities.select(
    "municipality_id", "name", "state_code", "region_name", "microregion_name", "immediate_region_name",
).orderBy("state_code").show(5, truncate=False)


## 5. População do IBGE

Extrai os municípios das linhas da planilha, monta o ID com os zeros necessários e converte a população para inteiro. A saída mantém todos os municípios da planilha, incluindo as capitais. As chamadas de nota ficam em `population_note`; o texto das notas continua disponível na Bronze. A data de referência e as origens também são preservadas.

Entrada: `data/bronze/ibge/population`  
Saída: `data/silver/ibge/population`

In [ ]:
# Recarrega o código salvo para não usar uma versão anterior mantida no kernel.
reload(ibge_population)

population = ibge_population.transform_population(
    spark,
    BRONZE_DIRECTORY / "ibge" / "population",
    SILVER_DIRECTORY / "ibge" / "population",
)

population.printSchema()
population.select(
    "municipality_id", "name", "state_code", "population_estimate", "reference_date", "population_note",
).orderBy("municipality_id").show(5, truncate=False)


## 6. Coordenadas das capitais

Padroniza nomes e UFs, valida IDs e coordenadas e reúne as origens dos registros equivalentes. Os limites de latitude e longitude não comprovam que o ponto está dentro do município.

Entrada: `data/bronze/open_meteo/geocoding`  
Saída: `data/silver/open_meteo/geocoding`

In [ ]:
# Recarrega o código salvo para não usar uma versão anterior mantida no kernel.
reload(geocoding)

coordinates = geocoding.transform_geocoding(
    spark,
    BRONZE_DIRECTORY / "open_meteo" / "geocoding",
    SILVER_DIRECTORY / "open_meteo" / "geocoding",
)

coordinates.printSchema()
coordinates.select(
    "municipality_id", "name", "state_code", "latitude", "longitude",
).orderBy("state_code").show(5, truncate=False)


## 7. Clima diário das capitais — Open-Meteo

Expande as listas em uma linha por município e data, validando as unidades e a cobertura do período declarado. As colunas numéricas indicam suas unidades: temperatura em °C, chuva em mm e vento em m/s. O vento recebido em km/h é dividido por 3,6. Valores ausentes ficam nulos, com identificação em `missing_fields`; alertas ficam em `quality_flags`.

Entrada: `data/bronze/open_meteo/weather/2025`  
Saída: `data/silver/open_meteo/weather/2025`  
As pastas de ano seguem o valor de `YEAR`.

In [ ]:
# Recarrega o código salvo para não usar uma versão anterior mantida no kernel.
reload(open_meteo)

weather = open_meteo.transform_open_meteo(
    spark,
    BRONZE_DIRECTORY / "open_meteo" / "weather" / YEAR,
    SILVER_DIRECTORY / "open_meteo" / "weather" / YEAR,
)

weather.printSchema()
weather.select(
    "name", "state_code", "observation_date", "temperature_mean_c", "precipitation_sum_mm", "wind_speed_max_ms", "invalid_fields", "quality_flags",
).orderBy("state_code", "observation_date").show(5, truncate=False)


## 8. Observações horárias — INMET

Trata os números com vírgula decimal e interpreta os horários UTC. Vazios e `-9999` viram nulos; zero de chuva e temperaturas negativas são preservados. Alertas físicos ficam sinalizados, sem excluir as linhas. Registros conflitantes para a mesma estação/hora interrompem a carga.

A transformação mantém todas as estações disponíveis, sem selecionar estações para cada capital ou agregar por dia. O vento continua em m/s, conforme a fonte; a Open-Meteo também usa m/s na Silver. A comparação ainda exige compatibilizar a velocidade horária e a rajada do INMET com o máximo diário da Open-Meteo.

Entrada: `data/bronze/inmet/2025`  
Saída: `data/silver/inmet/2025`  
As pastas de ano seguem o valor de `YEAR`.

A saída acrescenta `timezone`, `timestamp_local` (relógio local sem fuso embutido), `observation_date_local` e o deslocamento UTC do registro. O instante original permanece em `timestamp_utc`. Os campos `interval_start_utc`, `interval_end_utc`, `interval_start_local` e `interval_end_local` descrevem a hora anterior da medição. `interval_date_local` atribui uma hora terminada à meia-noite ao dia anterior, para futura agregação dos acumulados.

Referências: [precipitação no glossário do INMET](https://portal.inmet.gov.br/glossario/nublado), [TimezoneFinder](https://timezonefinder.readthedocs.io/en/latest/1_usage.html) e [conversão de fusos no Spark](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.convert_timezone.html).


O recorte `data/bronze/inmet/boundary_2025` contém somente as primeiras horas UTC de 01/01/2026, necessárias para fechar 31/12/2025 no horário local. A Silver correspondente fica em `data/silver/inmet/boundary_2025`; a Gold a usa automaticamente quando presente, mantendo a análise restrita a 2025.

In [ ]:
# Recarrega o código salvo para não usar uma versão anterior mantida no kernel.
reload(inmet)

station_observations = inmet.transform_inmet(
    spark,
    BRONZE_DIRECTORY / "inmet" / YEAR,
    SILVER_DIRECTORY / "inmet" / YEAR,
)

station_observations.printSchema()

# Mostra uma amostra sem ordenar a base horária inteira.
station_observations.select(
    "station_code", "state_code", "timezone", "timestamp_utc", "timestamp_local", "interval_date_local", "air_temperature_c", "precipitation_mm", "invalid_fields", "quality_flags",
).show(5, truncate=False)


## 9. Resumo da execução

As contagens abaixo são calculadas sobre os resultados desta execução. As fontes possuem granularidades diferentes: municípios, coordenadas, população por data de referência, clima diário e observações por estação/hora. Uma linha com ausência pode ser válida; um alerta é um ponto para investigação.

`missing_fields` identifica ausências; `invalid_fields` identifica as variáveis afetadas por erros numéricos ou regras físicas, sem alterar outras medições da linha. `quality_flags` explica as regras acionadas e `data_source` distingue as duas fontes de clima. Os limites de cobertura e os indicadores diários serão aplicados na Gold, sem preenchimento automático de lacunas.

In [ ]:
# Calcula os indicadores de clima em uma agregação por fonte.
weather_summary = weather.agg(
    F.count("*").alias("record_count"),
    F.sum(F.when(F.size("missing_fields") > 0, 1).otherwise(0)).alias("rows_with_missing"),
    F.sum(F.when(F.size("quality_flags") > 0, 1).otherwise(0)).alias("rows_with_flags"),
    F.sum(F.when(F.size("invalid_fields") > 0, 1).otherwise(0)).alias("rows_with_invalid"),
).first()

inmet_summary = station_observations.agg(
    F.count("*").alias("record_count"),
    F.countDistinct("station_code").alias("station_count"),
    F.sum(F.when(F.size("missing_fields") > 0, 1).otherwise(0)).alias("rows_with_missing"),
    F.sum(F.when(F.size("quality_flags") > 0, 1).otherwise(0)).alias("rows_with_flags"),
    F.sum(F.when(F.size("invalid_fields") > 0, 1).otherwise(0)).alias("rows_with_invalid"),
).first()

summary_records = [
    ("Municípios IBGE", municipalities.count()),
    ("População IBGE", population.count()),
    ("Geocoding", coordinates.count()),
    ("Open-Meteo diário", weather_summary["record_count"]),
    ("INMET horário", inmet_summary["record_count"]),
]

spark.createDataFrame(summary_records, "source string, record_count long").show(truncate=False)

print(f"Open-Meteo: {weather_summary['rows_with_missing']} linhas com ausências; {weather_summary['rows_with_flags']} linhas com alertas.")
print(f"INMET: {inmet_summary['station_count']} estações; {inmet_summary['rows_with_missing']} linhas com ausências; {inmet_summary['rows_with_flags']} linhas com alertas.")
print(f"Saídas da Silver: {SILVER_DIRECTORY}")
print(f"Open-Meteo: {weather_summary['rows_with_invalid']} linhas com variáveis inválidas.")
print(f"INMET: {inmet_summary['rows_with_invalid']} linhas com variáveis inválidas.")


## 10. Encerrar o Spark

Execute depois de terminar as transformações e as consultas. Para consultar os resultados em outra sessão, leia os Parquets com `recursiveFileLookup=True`, pois as pastas usam somente a sigla da UF.

## Tratar a borda UTC

Executa o mesmo tratamento da Silver sobre o recorte de seis horas. A Gold procura automaticamente a pasta irmã `boundary_2025`. Diferenças cadastrais superiores a 100 m ou de UF impedem usar a borda daquela estação e ficam identificadas em `boundary_metadata_status`.

In [ ]:
reload(inmet)

boundary_observations = inmet.transform_inmet(
    spark,
    BRONZE_DIRECTORY / "inmet" / f"boundary_{YEAR}",
    SILVER_DIRECTORY / "inmet" / f"boundary_{YEAR}",
)

boundary_observations.printSchema()


In [ ]:
execution.finish_notebook(session, spark)
